# CS781 Phase 5: Legal Statute Identification (LSI)
## Version 4: Lightweight Hybrid Decoder Transformer (HybridLegalLM + SwiGLU-LoRMA)
- **Backbone**: Phase 1 `HybridLegalLM` (12 Layers, 768 Hidden, GQA, RoPE, RMSNorm, 4 Global Layers [0, 4, 8, 11] + 8 Local Window Layers [256])
- **Adaptation**: Activation-space SwiGLU-LoRMA ($r=16, \alpha=32$) on attention projections + unfrozen final RMSNorm
- **Document Architecture**: Multi-Window Slicing (4 $\times$ 512 tokens) + Dual Window Pooling + Bidirectional Cross-Window Transformer + 100 Statute Query Attention Head
- **Loss**: Positive-weighted BCE Loss (mitigating 43:1 tail imbalance)
- **Calibration**: Support-regularized empirical threshold shrinkage
- **Output**: 100% Codabench-compliant `prediction.json` and `v4_hybrid_submission.zip`


In [ ]:
import os
import sys
import json
import math
import time
import random
import zipfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB)")


In [ ]:
# Hyperparameters & Paths
VOCAB_SIZE = 25000
N_EMBD = 768
N_LAYER = 12
N_HEAD = 12
N_KV_HEAD = 4
MAX_SEQ_LEN = 1024
FFN_MULTIPLIER = 4
DROPOUT = 0.0

GLOBAL_ATTENTION_LAYERS = [0, 4, 8, 11]
LOCAL_ATTN_WINDOW = 256

NUM_LABELS = 100
MAX_WINDOWS = 4
WINDOW_LEN = 512

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 16  # Effective batch size = 32
EPOCHS = 5
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

OUTPUT_DIR = "./phase5_v4_outputs"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
SUBMISSION_DIR = os.path.join(OUTPUT_DIR, "submissions")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)
print("Configuration verified. Effective batch size:", BATCH_SIZE * GRAD_ACCUM_STEPS)


In [ ]:
def find_file(accepted_names, env_var=None, required=True):
    if env_var and os.environ.get(env_var) and os.path.isfile(os.environ[env_var]):
        return os.environ[env_var]
    roots = ["/kaggle/input", "/mnt/data", ".", r"C:\Users\LENOVO\Downloads", r"D:\Phase5_LSI_Research"]
    accepted_lower = {name.lower() for name in accepted_names}
    candidates = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs if d not in {'.git', '.venv', 'venv', 'node_modules', '__pycache__'}]
            for f in files:
                if f.lower() in accepted_lower:
                    full_p = os.path.join(current, f)
                    if "phase_1" in full_p.lower() and "pretraining" in full_p.lower():
                        continue
                    candidates.append(full_p)
    if candidates:
        for name in accepted_names:
            for c in candidates:
                if os.path.basename(c).lower() == name.lower():
                    return c
        candidates.sort(key=lambda p: (len(Path(p).parts), len(p)))
        return candidates[0]
    if required:
        raise FileNotFoundError(f"Could not find any of: {accepted_names}. Please attach the dataset.")
    return None

CHECKPOINT_PATH = find_file(["hybrid_model.pt", "v4_model.pt"], env_var="LSI_V4_CHECKPOINT")
TOKENIZER_PATH = find_file(["bpe_en_25k.json"], env_var="LSI_TOKENIZER")
TRAIN_PATH = find_file(["train (2).json", "train (2)(1).json", "train.json"], env_var="LSI_TRAIN_JSON")
DEV_PATH = find_file(["dev (1).json", "dev.json"], env_var="LSI_DEV_JSON")
TEST_PATH = find_file(["test (2).json", "test.json"], env_var="LSI_TEST_JSON")

print("Discovered files:")
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Tokenizer:  {TOKENIZER_PATH}")
print(f"  Train:      {TRAIN_PATH}")
print(f"  Dev:        {DEV_PATH}")
print(f"  Test:       {TEST_PATH}")


In [ ]:
with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_stoi = {token: index for index, token in enumerate(bpe_itos)}
merge_to_id = {(bpe_stoi[m[0]], bpe_stoi[m[1]]): 261 + i for i, m in enumerate(bpe_merges)}

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.vocab_size = len(itos)
        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        seq = [4 + b for b in word.encode("utf-8")] + [self.word_boundary_id]
        while len(seq) > 1:
            best_pos, best_id = None, None
            for i in range(len(seq) - 1):
                pair = (seq[i], seq[i + 1])
                t_id = self.merge_to_id.get(pair)
                if t_id is not None and (best_id is None or t_id < best_id):
                    best_id, best_pos = t_id, i
            if best_pos is None:
                break
            seq = seq[:best_pos] + [best_id] + seq[best_pos + 2:]
        return seq

    def encode(self, text, add_bos=False, add_eos=False):
        tokens = [self.bos_token_id] if add_bos else []
        for word in text.split():
            tokens.extend(self._encode_word(word))
        if add_eos:
            tokens.append(self.eos_token_id)
        return tokens

tokenizer = ByteLevelBPE25k(bpe_itos, bpe_stoi, merge_to_id)
print("Tokenizer loaded. Vocab size:", tokenizer.vocab_size)


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(rms + self.eps))

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000):
        super().__init__()
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)
    def _build_cache(self, seq_len):
        positions = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(positions, self.inv_freq)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
    def forward(self, q, k, position_offset=0):
        seq_len = q.size(-2)
        required_len = position_offset + seq_len
        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len
        cos = self.cos_cached[position_offset: position_offset + seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[position_offset: position_offset + seq_len].unsqueeze(0).unsqueeze(0)
        q = self._apply_rotary(q, cos, sin)
        k = self._apply_rotary(k, cos, sin)
        return q, k
    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        rotated = torch.stack((-x2, x1), dim=-1).flatten(-2)
        cos_full = torch.repeat_interleave(cos, 2, dim=-1)
        sin_full = torch.repeat_interleave(sin, 2, dim=-1)
        return x * cos_full + rotated * sin_full

class GQAAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, max_seq_len, attention_type="global", local_window=256, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head
        self.kv_repeat = n_head // n_kv_head
        self.attention_type = attention_type
        self.local_window = local_window
        self.dropout = dropout
        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.rope = RotaryEmbedding(head_dim=self.head_dim, max_seq_len=max_seq_len)
    def _build_attention_mask(self, q_len, k_len, device):
        row_indices = torch.arange(q_len, device=device).unsqueeze(1)
        col_indices = torch.arange(k_len, device=device).unsqueeze(0)
        causal_mask = col_indices <= row_indices
        if self.attention_type == "local":
            local_mask = col_indices >= (row_indices - self.local_window + 1)
            final_mask = causal_mask & local_mask
        else:
            final_mask = causal_mask
        return final_mask
    def forward(self, x, past_key_value=None, use_cache=False):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        offset = past_key_value[0].size(-2) if past_key_value is not None else 0
        q, k = self.rope(q, k, position_offset=offset)
        if past_key_value is not None:
            k = torch.cat([past_key_value[0], k], dim=-2)
            v = torch.cat([past_key_value[1], v], dim=-2)
        present = (k, v) if use_cache else None
        if self.kv_repeat > 1:
            k = torch.repeat_interleave(k, self.kv_repeat, dim=1)
            v = torch.repeat_interleave(v, self.kv_repeat, dim=1)
        q_len = q.size(-2)
        k_len = k.size(-2)
        attn_mask = self._build_attention_mask(q_len, k_len, x.device)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0, is_causal=False)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y), present

class FeedForward(nn.Module):
    def __init__(self, n_embd, multiplier=4, dropout=0.0):
        super().__init__()
        hidden_dim = int(n_embd * multiplier)
        self.fc1 = nn.Linear(n_embd, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, max_seq_len, ffn_multiplier, dropout, layer_index, global_attention_layers, local_window):
        super().__init__()
        self.layer_index = layer_index
        attention_type = "global" if layer_index in global_attention_layers else "local"
        self.attention_type = attention_type
        self.norm1 = RMSNorm(n_embd)
        self.attention = GQAAttention(n_embd=n_embd, n_head=n_head, n_kv_head=n_kv_head, max_seq_len=max_seq_len, attention_type=attention_type, local_window=local_window, dropout=dropout)
        self.norm2 = RMSNorm(n_embd)
        self.ffn = FeedForward(n_embd=n_embd, multiplier=ffn_multiplier, dropout=dropout)
    def forward(self, x, past_key_value=None, use_cache=False):
        attn_output, present = self.attention(self.norm1(x), past_key_value=past_key_value, use_cache=use_cache)
        x = x + attn_output
        x = x + self.ffn(self.norm2(x))
        return x, present

class HybridLegalLM(nn.Module):
    def __init__(self, vocab_size=25000, n_embd=768, n_layer=12, n_head=12, n_kv_head=4, max_seq_len=1024, ffn_multiplier=4, dropout=0.0, global_attention_layers=[0, 4, 8, 11], local_window=256):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([
            TransformerBlock(n_embd, n_head, n_kv_head, max_seq_len, ffn_multiplier, dropout, layer_index=i, global_attention_layers=global_attention_layers, local_window=local_window)
            for i in range(n_layer)
        ])
        self.final_norm = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        
    def extract_features(self, input_ids):
        x = self.token_embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x)
        return self.final_norm(x)

print("HybridLegalLM backbone class defined.")


In [ ]:
class SwiGLULoRMALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.rank = rank
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)
        d_out = base.out_features
        self.A_gate = nn.Linear(d_out, rank, bias=False)
        self.A_lin = nn.Linear(d_out, rank, bias=False)
        self.A_glob_gate = nn.Linear(d_out, rank, bias=False)
        self.A_glob_lin = nn.Linear(d_out, rank, bias=False)
        self.B = nn.Linear(rank, d_out, bias=False)
        nn.init.kaiming_uniform_(self.A_gate.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.A_lin.weight, a=math.sqrt(5))
        nn.init.zeros_(self.A_glob_gate.weight)
        nn.init.zeros_(self.A_glob_lin.weight)
        nn.init.zeros_(self.B.weight)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        z = self.base(x)
        g = z.mean(dim=1, keepdim=True)
        z_drop = self.dropout(z)
        u = self.A_gate(z_drop) + self.A_glob_gate(g)
        v = self.A_lin(z_drop) + self.A_glob_lin(g)
        delta = self.B(F.silu(u) * v) * self.scale
        return z + delta

def apply_lorma_to_hybrid(model: HybridLegalLM, rank=16, alpha=32.0, dropout=0.05):
    for p in model.parameters():
        p.requires_grad_(False)
    for block in model.blocks:
        attn = block.attention
        attn.q_proj = SwiGLULoRMALinear(attn.q_proj, rank=rank, alpha=alpha, dropout=dropout)
        attn.v_proj = SwiGLULoRMALinear(attn.v_proj, rank=rank, alpha=alpha, dropout=dropout)
    model.final_norm.weight.requires_grad_(True)

class WindowDualPooling(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.attn_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.fuse_proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, token_reps: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask_f = mask.unsqueeze(-1).to(token_reps.dtype)
        denom = mask_f.sum(dim=1).clamp_min(1.0)
        mean_pooled = (token_reps * mask_f).sum(dim=1) / denom
        scores = self.attn_proj(token_reps)
        scores = scores.masked_fill(~mask.unsqueeze(-1).bool(), float("-inf"))
        attn_weights = F.softmax(scores, dim=1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        attn_pooled = (token_reps * attn_weights).sum(dim=1)
        return self.fuse_proj(torch.cat([mean_pooled, attn_pooled], dim=-1))

class CrossWindowTransformer(nn.Module):
    def __init__(self, hidden_dim: int = 768, num_layers: int = 2, num_heads: int = 8, max_windows: int = 16, dropout: float = 0.1):
        super().__init__()
        self.pos_emb = nn.Embedding(max_windows, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=hidden_dim * 2, dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
    def forward(self, window_embeds: torch.Tensor, window_mask: torch.Tensor) -> torch.Tensor:
        B, W, D = window_embeds.size()
        positions = torch.arange(W, device=window_embeds.device).unsqueeze(0).expand(B, W)
        x = window_embeds + self.pos_emb(positions)
        src_key_padding_mask = ~window_mask.bool()
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)

class LabelQueryEvidenceAttention(nn.Module):
    def __init__(self, num_labels: int = 100, hidden_dim: int = 768, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.num_labels = num_labels
        self.label_queries = nn.Parameter(torch.randn(num_labels, hidden_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, window_context: torch.Tensor, window_mask: torch.Tensor) -> torch.Tensor:
        B, W, D = window_context.size()
        queries = self.label_queries.unsqueeze(0).expand(B, self.num_labels, D)
        key_padding_mask = ~window_mask.bool()
        attn_out, _ = self.cross_attn(query=queries, key=window_context, value=window_context, key_padding_mask=key_padding_mask)
        label_reps = self.norm(queries + attn_out)
        return self.classifier(label_reps).squeeze(-1)

class HLGHybridLSI(nn.Module):
    def __init__(self, backbone: HybridLegalLM, num_labels: int = 100, hidden_dim: int = 768, lorma_rank: int = 16, lorma_alpha: float = 32.0, max_windows: int = 4):
        super().__init__()
        self.backbone = backbone
        apply_lorma_to_hybrid(self.backbone, rank=lorma_rank, alpha=lorma_alpha)
        self.window_pooling = WindowDualPooling(hidden_dim)
        self.cross_window_transformer = CrossWindowTransformer(hidden_dim=hidden_dim, max_windows=max_windows)
        self.head = LabelQueryEvidenceAttention(num_labels=num_labels, hidden_dim=hidden_dim)

    def forward(self, window_input_ids: torch.Tensor, token_mask: torch.Tensor, window_mask: torch.Tensor) -> torch.Tensor:
        B, W, L = window_input_ids.size()
        flat_input_ids = window_input_ids.view(B * W, L)
        flat_token_mask = token_mask.view(B * W, L)
        flat_token_reps = self.backbone.extract_features(flat_input_ids)
        flat_window_embeds = self.window_pooling(flat_token_reps, flat_token_mask)
        window_embeds = flat_window_embeds.view(B, W, -1)
        window_context = self.cross_window_transformer(window_embeds, window_mask)
        return self.head(window_context, window_mask)

print("HLG-Hybrid-LSI architecture defined.")


In [ ]:
class LSIDataset(Dataset):
    def __init__(self, json_path, tokenizer, max_windows=4, window_len=512, is_test=False):
        self.tokenizer = tokenizer
        self.max_windows = max_windows
        self.window_len = window_len
        self.is_test = is_test
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        print(f"Loaded {len(self.data):,} items from {json_path}")

    def __len__(self):
        return len(self.data)

    def _create_windows(self, token_ids):
        L = self.window_len
        W = self.max_windows
        if len(token_ids) <= L:
            chunks = [token_ids]
        else:
            total = len(token_ids)
            starts = [0]
            step = (total - L) / max(1, W - 1)
            for i in range(1, W - 1):
                starts.append(int(round(i * step)))
            starts.append(total - L)
            chunks = [token_ids[s : s + L] for s in starts]

        windows, masks = [], []
        for c in chunks[:W]:
            toks = c[:L]
            mask = [1] * len(toks)
            if len(toks) < L:
                pad_len = L - len(toks)
                toks += [self.tokenizer.pad_token_id] * pad_len
                mask += [0] * pad_len
            windows.append(toks)
            masks.append(mask)

        valid_windows = len(windows)
        while len(windows) < W:
            windows.append([self.tokenizer.pad_token_id] * L)
            masks.append([0] * L)
        w_mask = [1] * valid_windows + [0] * (W - valid_windows)
        return torch.tensor(windows, dtype=torch.long), torch.tensor(masks, dtype=torch.float32), torch.tensor(w_mask, dtype=torch.float32)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = self.tokenizer.encode(item["text"])
        w_ids, t_mask, w_mask = self._create_windows(tokens)
        if self.is_test:
            return {"id": item["id"], "w_ids": w_ids, "t_mask": t_mask, "w_mask": w_mask}
        target = torch.zeros(100, dtype=torch.float32)
        target[item.get("labels", [])] = 1.0
        return {"id": item["id"], "w_ids": w_ids, "t_mask": t_mask, "w_mask": w_mask, "target": target}

train_dataset = LSIDataset(TRAIN_PATH, tokenizer, max_windows=MAX_WINDOWS, window_len=WINDOW_LEN)
dev_dataset = LSIDataset(DEV_PATH, tokenizer, max_windows=MAX_WINDOWS, window_len=WINDOW_LEN)
test_dataset = LSIDataset(TEST_PATH, tokenizer, max_windows=MAX_WINDOWS, window_len=WINDOW_LEN, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
print(f"Loading Base HybridLegalLM from {CHECKPOINT_PATH}...")
base_model = HybridLegalLM(
    vocab_size=VOCAB_SIZE, n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD,
    max_seq_len=MAX_SEQ_LEN, ffn_multiplier=FFN_MULTIPLIER, dropout=DROPOUT,
    global_attention_layers=GLOBAL_ATTENTION_LAYERS, local_window=LOCAL_ATTN_WINDOW
)
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
sd = ckpt.get("model_state_dict", ckpt)
base_model.load_state_dict(sd, strict=True)
print("Base model loaded with 100% strict matching.")

model = HLGHybridLSI(
    backbone=base_model, num_labels=NUM_LABELS, hidden_dim=N_EMBD,
    lorma_rank=16, lorma_alpha=32.0, max_windows=MAX_WINDOWS
)
model.to(DEVICE)

trainable_params = [p for p in model.parameters() if p.requires_grad]
total_params = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in trainable_params)
print(f"Model ready. Trainable params: {trainable_p:,} / {total_params:,} ({trainable_p / total_params * 100:.2f}%)")

# Positive-weighted BCE Loss
all_targets = np.zeros((len(train_dataset), NUM_LABELS), dtype=np.float32)
for i in range(len(train_dataset)):
    all_targets[i, train_dataset.data[i].get("labels", [])] = 1.0
pos_counts = all_targets.sum(axis=0)
pos_weights = np.sqrt((len(all_targets) - pos_counts) / np.maximum(pos_counts, 1.0))
pos_weights = np.clip(pos_weights, 1.0, 8.0)
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
print("Positive-weighted BCE Loss configured.")


In [ ]:
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def calibrate_thresholds(y_true, y_probs):
    calib_th = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0:
            continue
        best_f, best_t = -1.0, 0.5
        for t in np.arange(0.04, 0.50, 0.02):
            f = f1_score(y_true[:, c], (y_probs[:, c] >= t).astype(int), zero_division=0)
            if f > best_f:
                best_f, best_t = f, t
        calib_th[c] = best_t
    return calib_th

best_macro_f1 = -1.0
best_checkpoint_path = os.path.join(CHECKPOINT_DIR, "v4_hybrid_best_model.pt")

print(f"Starting {EPOCHS}-Epoch Hybrid Training...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for step, batch in enumerate(pbar):
        w_ids = batch["w_ids"].to(DEVICE, non_blocking=True)
        t_mask = batch["t_mask"].to(DEVICE, non_blocking=True)
        w_mask = batch["w_mask"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits = model(w_ids, t_mask, w_mask)
            loss = criterion(logits, targets) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        pbar.set_postfix({"loss": f"{loss.item() * GRAD_ACCUM_STEPS:.4f}"})

    # Validation
    model.eval()
    dev_probs, dev_targets = [], []
    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch}/{EPOCHS} [Dev]"):
            w_ids = batch["w_ids"].to(DEVICE, non_blocking=True)
            t_mask = batch["t_mask"].to(DEVICE, non_blocking=True)
            w_mask = batch["w_mask"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                probs = torch.sigmoid(model(w_ids, t_mask, w_mask))
            dev_probs.append(probs.cpu().numpy())
            dev_targets.append(batch["target"].numpy())

    dev_probs = np.concatenate(dev_probs, axis=0)
    dev_targets = np.concatenate(dev_targets, axis=0)

    calib_th = calibrate_thresholds(dev_targets, dev_probs)
    macro_f1 = f1_score(dev_targets, (dev_probs >= calib_th).astype(int), average="macro", zero_division=0)
    micro_f1 = f1_score(dev_targets, (dev_probs >= calib_th).astype(int), average="micro", zero_division=0)
    print(f"Epoch {epoch} Complete. Dev Macro-F1 (Calibrated): {macro_f1:.6f} | Micro-F1: {micro_f1:.6f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_macro_f1": best_macro_f1,
            "calibrated_thresholds": calib_th.tolist()
        }, best_checkpoint_path)
        print(f"Saved Best Checkpoint (Macro-F1: {best_macro_f1:.6f})")


In [ ]:
print(f"Loading Best Model from {best_checkpoint_path}...")
best_ckpt = torch.load(best_checkpoint_path, map_location=DEVICE)
model.load_state_dict(best_ckpt["model_state_dict"])
calibrated_th = np.array(best_ckpt["calibrated_thresholds"], dtype=np.float32)
print(f"Loaded Best Model (Dev Macro-F1: {best_ckpt['best_macro_f1']:.6f})")

model.eval()
test_predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating Test Predictions"):
        ids = batch["id"]
        w_ids = batch["w_ids"].to(DEVICE, non_blocking=True)
        t_mask = batch["t_mask"].to(DEVICE, non_blocking=True)
        w_mask = batch["w_mask"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            probs = torch.sigmoid(model(w_ids, t_mask, w_mask)).cpu().numpy()

        for doc_id, prob in zip(ids, probs):
            pred_indices = np.where(prob >= calibrated_th)[0].tolist()
            if len(pred_indices) == 0:
                pred_indices = [int(np.argmax(prob))]
            test_predictions.append({"id": str(doc_id), "pred": pred_indices})

print(f"Generated {len(test_predictions):,} test predictions.")
assert len(test_predictions) == len(test_dataset), f"Count mismatch: {len(test_predictions)} != {len(test_dataset)}"

pred_json_path = os.path.join(SUBMISSION_DIR, "prediction.json")
with open(pred_json_path, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f)
print(f"Saved: {pred_json_path}")

submission_zip_path = os.path.join(SUBMISSION_DIR, "v4_hybrid_submission.zip")
with zipfile.ZipFile(submission_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_json_path, arcname="prediction.json")
print(f"Packaged Submission ZIP: {submission_zip_path} ({os.path.getsize(submission_zip_path):,} bytes)")

# Sanity check
with open(pred_json_path, "r", encoding="utf-8") as f:
    chk = json.load(f)
assert isinstance(chk, list), "prediction.json must be a list"
assert len(chk) == 13019, f"Expected 13,019 items, got {len(chk)}"
for r in chk[:100]:
    assert isinstance(r, dict) and "id" in r and "pred" in r
    assert all(isinstance(x, int) and 0 <= x < 100 for x in r["pred"])
print("="*65)
print("V4 HYBRID SUBMISSION CHECKS PASSED (100% CODABENCH COMPLIANT).")
print(f"Ready for submission: {submission_zip_path}")
print("="*65)
